# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [11]:

from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [12]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [13]:
# atom() is provided in sudoku_solver.py and imported above.
atom('Not', 2,2,4)
atom('Is', 2,2,4)

Is2_2_4

## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [14]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.



box_h = 3
box_w = 3
n = 9
kb = build_general_kb(n, box_h, box_w, givens)





print(len(kb.clauses))


11775


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [27]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
general_kb = build_general_kb(n, box_h, box_w, givens)
r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
query = atom('Is', r, c, v)
#
print(pl_resolution(general_kb, query))    
# print(tt_entails(associate('&', general_kb.clauses), query)) 

KeyboardInterrupt: 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


Both are taking too long to completed
tt_entails approach require all scnearios to be mapped and in a 9x9 puzzle, there is 9x9x9=720 symbols
A truth table will require 2^720 possible assignment, which my calculator cannot compute. With the current compute power, the answer will never come.

for PL_resolution, it negates a query to look for contradiction. this also need to run all rules to reach a conclusion.
in the KB there are more than 10,000 clauses and to pair it with negation will require huge memory to be able to reach a contradiction.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [15]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.
import time
def print_grid(solution, n=9, box_h=3, box_w=3):
    print("+" + "---+" * n)
    for r in range(1, n + 1): 
        row_str = "|" 
        for c in range(1, n + 1): 
            val = solution.get((r, c), ".") 
            row_str += f" {val} "
            if c % box_w == 0: 
                row_str += "|"
            else: 
                row_str += " "
        print(row_str)
        if r % box_h == 0:
            print("+" + "---+" * n)


#(b) Forward chaining on full grid
start_time = time.time()
solution_fc = solve_full_grid_fc(n, box_h, box_w, givens)
elapsed_time = time.time() - start_time
print(f"Puzzle solved with Forward Chaining in {elapsed_time:.4f} seconds!\n") 
print_grid(solution_fc, n, box_h, box_w)

#(c) Backward chaining on full grid
start_time = time.time()
solution_bc = solve_full_grid_bc(n, box_h, box_w, givens)
elapsed_time = time.time() - start_time
print(f"Puzzle solved with Backward Chaining in {elapsed_time:.4f} seconds!\n") 
print_grid(solution_bc, n, box_h, box_w)


Puzzle solved with Forward Chaining in 16.4274 seconds!

+---+---+---+---+---+---+---+---+---+
| 1   3   6 | 9   8   5 | 4   7   2 |
| 2   9   4 | 3   1   7 | 8   6   5 |
| 5   8   7 | 2   6   4 | 1   3   9 |
+---+---+---+---+---+---+---+---+---+
| 7   5   1 | 4   9   6 | 2   8   3 |
| 3   6   9 | 8   7   2 | 5   1   4 |
| 4   2   8 | 5   3   1 | 7   9   6 |
+---+---+---+---+---+---+---+---+---+
| 8   1   2 | 6   4   9 | 3   5   7 |
| 6   4   3 | 7   5   8 | 9   2   1 |
| 9   7   5 | 1   2   3 | 6   4   8 |
+---+---+---+---+---+---+---+---+---+
Puzzle solved with Backward Chaining in 1.1968 seconds!

+---+---+---+---+---+---+---+---+---+
| 1   3   6 | 9   8   5 | 4   7   2 |
| 2   9   4 | 3   1   7 | 8   6   5 |
| 5   8   7 | 2   6   4 | 1   3   9 |
+---+---+---+---+---+---+---+---+---+
| 7   5   1 | 4   9   6 | 2   8   3 |
| 3   6   9 | 8   7   2 | 5   1   4 |
| 4   2   8 | 5   3   1 | 7   9   6 |
+---+---+---+---+---+---+---+---+---+
| 8   1   2 | 6   4   9 | 3   5   7 |
| 6   4   3 

### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [16]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time
#
t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")

solve_full_grid_fc: 17.44s
solve_full_grid_bc: 1.18s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

For General KB there are some of my key points I started off writing rules for each of the sudoku condition. Iterating each cell only using "Is_r_c_v" combination to check for the condition.

#1 first rule for each cell to have a value is the most unique setup compared to other rules. i use a longer disjunctive CNF clause containing nine positive literals, to ensure at least one of the values 1-9 is true. This has to be enforced with condition #2 to enforce exactly one value per cell.

    The rule simply checks that at least 1 value is true for each cell: (Is_r_c_1 | Is_r_c_2 | Is_r_c_3 ....)


#2 All the other conditions, i use pairwise binary negative clauses to enforce the rule. This simplified approach is to check each pair of condition for collision (Both Is_1_1_1 and Is_1_1_2 must not be true at the same time).
Condition #1 does not directly enforce every row contains every number; it only enforces every cell to have at least one value. Condition #2 then enforces at most one value per cell. Therefore, conditions #1 and #2 together enforce exactly one value per cell.

    The smae logical pattern i used but on different propositions:
    - Within one cell: (~Is_1_1_1 | ~Is_1_1_2) → cell (1,1) cannot be both 1 and 2.
    - Across one row: (~Is_1_1_1 | ~Is_1_2_1) → cells (1,1) and (1,2) cannot both contain 1.

#3 In my original approach, the orinial idea is to reduce the observation of overalapping/repeating premise being used. So i refactor to a "Walk forward" approach to reduce the number of loops and rules needed. 
    
    For example while iteration the pair across columns (~Is_r_c1_1 | ~Is_r_c2_1) c2 will always start from c1+1 and only iterate forward, preventing reversed version of the same pair. Each iteration on c1 will take n-c1 number of rules to build into the kb. 
    
    c1 = 1 will have 8 pairs of rules for the same value
    c1 = 2 will have 7 pairs of rules for the same value
    c1 = 3 will have 6 pairs of rules for the same value
    ...
    This ensures each pair is compared only once and prevents the reversed version of the same pair from generating a duplicate rule.

#4 In my original approach, for the rule on no two cells in the same 3x3 box can contain the same value, i iterate the starting point to be jump cell with step 3 on rows on outer loop and column as inner loop. Then i created a liner representaion of the 3x3 box sequence to make the loop function easier. After that the concept is the same for how unique values is compared across rows or columns.

#5 I eventually optimised the python code with to combine conditions 3-5, refactoring the above points #3 and #4 rather than treating row, column and box uniqueness as 3 seperate algo, i combined what each of the condition is trying to do (2 cells do not share same value). So i change to use the concept of peers relationship and because each cell can be peers to many other cells, the set in peers form the multiple relationship to optimise looping for these conditions.  There is one more puzzle piece of this final optimisation where the seen set prevents unordered pair from being repeatedly added to the KB.
    - Peers uses a set to prevent same cell from appearing multiple times as a perr when relationships overlap
    - seen prevents same unordered pair of cells from being processed twice

#6 Lastly, the simplest condition was #6 of the givens where (1,2) = 3 becomes Is_1_2_3. The givens are added as positive unti clauses (facts).

The above resulted in each constraint represented as a clause containing a disjunction of literals and the entire KB is the conjunction chaining up of all these clauses, therefore CNF. 




For the Definite KB, I cannot directly reuse all the approach from the General KB because definite/Horn clauses have a more restricted structure. In particular, the General KB can directly use negation such as ~Is_r_c_v and a disjunction such as (Is_r_c_1 | Is_r_c_2 | ... | Is_r_c_9), while these cannot be directly represented as definite clauses.

To work around this restriction, the Not_r_c_v propositions has to be introduced. Not_r_c_v is a separate positive proposition meaning that value v has been eliminated from cell (r,c). It is not the logical negation ~Is_r_c_v. This allows elimination to be represented as the positive conclusion of a definite rule using ==>. The overall idea is to use Is and Not together in an elimination cycle, where Is ==> Not eliminates candidates and accumulated Not facts can eventually infer a new Is.

#1 using a known Is_r_c_v as an anchor to eliminate all other possible values from the same cell to satisfy condition #2 at most one value in every cell. The elmination below shows that once a cell is known to contain value 1, all the other values for the same cell are inferred as "Not".

    Is_1_1_1 ==> Not_1_1_2
    Is_1_1_1 ==> Not_1_1_3
    ...

#2 Applying the same concept on Is ===> Not elimination concept for conditions #3-#5, instead of eliminating a different value from the same cell like in #1 above, the known value is eliminated from all of the cell's peers. Similar to my final GeneralKB approack, i reuse the peers concept instead of looping row, column and box constraints seperately as they all can be represented using the same type of elimination rule

    for example, if a cell already has value 5, its peer cannot be value 5
    Is_1_1_5 ==> Not_1_2_5
    Is_1_1_5 ==> Not_2_1_5

#3 going back to condition #1, it creates a different problem for DefiniteKB. I cannot use the same approach like GeneralKB to represent at least one value using disjunction of all possible value of a cell (Is_r_c_1 | Is_r_c_2 | ... | Is_r_c_9). This is not definite clause due to multiple positive proposition. Therefore using last-candidate rule where 8 possible values of a cell have already been eliminate, the remaining value can be inferred as a value of cell. 

    to infer value = 5
    (Not_r_c_1 & Not_r_c_2 & ... Not_r_c_9) ==> Is_r_c_5

#4 givens in condition #6 is simplified as fact. These are important initial facts because they provide the starting Is propositions from which the elimination rules can begin firing. Is_r_c_v is already a definate clause. 


Overall, comparing with GeneralKB, the key difference is i can no longer express using unrestricted disjunction and negation. The constraints need to be directional elimination rules using posivitve propositions. It can therefore be summarised into 2 main types of inference:

    To eliminate possible candidates:
    Is_r_c_v ==> Not_r_c_otherV / Not_peerR_peerC_v

    To infer last remaining candidate:
    Not_r_c_v1 & Not_r_c_v2 & ... ==> Is_r_c_remainingV


Lastly, the conditions appear in a different order in my Definite KB mainly because of how I structured and optimised the rule-generation loops. This construction order does not impose an execution order on the KB. Once the KB has been built, all elimination and last-candidate rules are available to the inference algorithm.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

Both truth-table model checking (tt_entails) and resolution-refutation (pl_resolution) are sound and complete for propositional logic. Therefore, theoretically they can determine whether a Sudoku query is entailed by the General KB. However, I would not use either of them as the default algorithm for solving Sudoku because their computational cost becomes extremely large.

Truth-table model checking:

For tt_entails, the main problem is the size of the possible model space. A 9×9 Sudoku using the General KB has 9 × 9 × 9 = 729 Is_r_c_v propositional symbols.

Each proposition can independently be either True or False when generating possible truth assignments. Therefore, in the worst case, the algorithm may need to explore 2^729 possible truth assignments.

Although most of these assignments do not represent valid Sudoku states, truth-table model checking still has to explore the possible models and test whether the KB and query hold. Therefore, the exponential growth makes truth-table entailment computationally impractical for a full Sudoku KB despite its theoretical completeness.

Resolution-refutation: 

Resolution-refutation approaches the problem differently. To determine whether the KB entails a query Q, it adds the negated query ~Q to the KB and attempts to derive a contradiction through complementary literals. If resolution eventually derives the empty clause (False), then KB ∧ ~Q is inconsistent and therefore KB |= Q.

The problem is that resolution repeatedly considers pairs of CNF clauses and generates new resolvents from complementary literals. These newly generated clauses are then added back into the clause set and can participate in further resolution steps.

Therefore, as the number of clauses increases, the number of possible clause pairs and generated resolvents can grow very rapidly. This results in large computational and memory requirements even though resolution is sound and complete.

Conclusion

Therefore, theoretical completeness does not imply computational tractability. Both truth-table model checking and resolution-refutation are useful general-purpose propositional inference algorithms, but I would not use either as the default Sudoku solver.

Truth-table checking suffers from the exponential 2^729 model space, while resolution can suffer from rapid growth in the number of clauses and pairwise resolution operations. For Sudoku, the more structured Definite/Horn representation allows forward or backward chaining to exploit the directional elimination rules without exhaustively considering all possible truth assignments or arbitrary pairs of clauses.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**

BACKWARD-CHAIN(KB, query)

Extract all known facts from KB
Group rules according to their conclusion
Add known facts to PROVED

REPEAT
    Record current number of PROVED facts
    Clear FAILED-THIS-PASS
    Clear ACTIVE goals

    IF PROVE(query) succeeds
        RETURN True

    IF no new facts were added to PROVED
        RETURN False

PROVE(goal)

IF goal is already in PROVED
    RETURN True

IF goal is already ACTIVE
    RETURN False

IF goal is in FAILED-THIS-PASS
    RETURN False

Mark goal as ACTIVE

FOR each rule whose conclusion matches goal
    Attempt to PROVE every premise of the rule
    IF ALL premises are successfully proved
        Add goal to PROVED
        Remove goal from ACTIVE
        RETURN True

Remove goal from ACTIVE
Add goal to FAILED-THIS-PASS

RETURN False



Facts are the base case → premises become recursive subgoals → AND within a rule / OR between rules → cycle detection and caching guarantee termination.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**

For my Definite KB, the current elimination rules mainly reason using a known Is value to produce Not eliminations, followed by the last-candidate rule to infer a new Is. More advanced Sudoku techniques can also be represented using definite/Horn clauses, but doing so requires more complex and specialised rules.

Using Naked Pair technique as comparison example:
A Naked Pair occurs when two cells within the same row, column, or box have exactly the same two remaining candidate values.

    For example, suppose two cells in row 1 have only values 2 and 7 remaining:
    Cell (1,1): {2,7}
    Cell (1,5): {2,7}

Since these two cells must contain 2 and 7 in some order, no other cell within the same row can contain them.


#2 Representing the Naked Pair using Not
My existing Not_r_c_v propositions can be used to identify which candidates have already been eliminated.
For cell (1,1) to contain only candidates {2,7}, all values other than 2 and 7 must already be Not:

    Not_1_1_1 & Not_1_1_3 & Not_1_1_4 &
    Not_1_1_5 & Not_1_1_6 & Not_1_1_8 & Not_1_1_9

The same must be true for cell (1,5):

    Not_1_5_1 & Not_1_5_3 & Not_1_5_4 &
    Not_1_5_5 & Not_1_5_6 & Not_1_5_8 & Not_1_5_9

When both conditions are satisfied, the two cells form the Naked Pair {2,7}.
I can then construct definite rules to eliminate these values from another cell in the same unit. For example, for cell (1,8):


#3 Why Horn logic can represent this technique
I do not need to determine whether (1,1) is 2 and (1,5) is 7, or vice versa. The Naked Pair only allows me to conclude that other cells in the same unit cannot contain 2 or 7. Since Not_r_c_v is represented as a positive proposition in my Definite KB, each of these eliminations can be used as the single conclusion of a definite rule. Therefore, the Naked Pair reasoning can still be represented using the same Is and Not vocabulary already used by my Definite KB.



A Naked Pair rule requires many premises to establish that two cells each have exactly the same two remaining candidates. Rules would also have to be generated for many combinations of cell pairs, candidate-value pairs, and the other cells within each row, column, or box.

Therefore, adding Naked Pair rules would increase the deductive power of the Definite KB and could allow puzzles requiring more advanced elimination to be solved. However, this comes at the cost of a much larger KB, more rule-generation work, and potentially more work for forward or backward chaining.

This shows that the limitation of using Horn logic is not necessarily that advanced Sudoku techniques are impossible to represent. Rather, increasingly sophisticated reasoning may require a much larger and more specialised collection of definite rules.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**

a) Backwards chaining in this example is goal-driven where demostrated in my solution, only explore rules that contributes to the goal.

For example, to query if a cell has value 5, Is_r_c_5, this can be proved from local sequence of elimination where only the related query and premises are evaluated. The other rules that are not related to this will not be explored. This approach is different from running thru all the rules in the KB for answer. This is also demostrated in Part A - Verifying the algo, Forward chaining took about 17 seconds while backwards chaining took 1.25 seconds to complete the same task.

b) Backwards chaining performance gain from skipping unrelated rules, when a query has to related to most of the rules within the KB, the performance gain comparing from forward chaining might diminish. The reason is because more query will be added as it explore each rule in the KB and may have to run deeper into every path. A good example shall be when little to no givens are provided to the KB, where it need to run almost every rules to solve the puzzle.

While optimisation demostrated in my solution may be able to improve backwards chaining from performing repetead work, forward chaining might be a better approach from known facts deriving consequences. These know consequences can also be shared with multiple queries for improvements.

Therefore the choise of using forward or backward chaining depends largely on the KB, query workload and implementation. 


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://it5005-grp6-sudoku-app.streamlit.app`
